Successfully import libraries for Life Expectancy df cleaning

In [105]:
import pandas as pd
import numpy as np
from thefuzz import process

Successfully loaded the Life_Expectancy_Data.csv as a Pandas dataframe in variable life_expectancy_raw, and copied dataframe to new variable life_expectancy.

In [106]:
life_expectancy_raw = pd.read_csv('../Datasets/Raw_datasets/Life_Expectancy_Data.csv')
life_expectancy = life_expectancy_raw.copy()


Successfully loaded the world_bank_population.csv as a Pandas dataframe in variable world_population.

In [107]:
world_population = pd.read_csv('../Datasets/Raw_datasets/Population/world_bank_population.csv', on_bad_lines='skip')


Successfully loaded the world_bank_gdp_1960_to_2023.csv as a Pandas dataframe in variable wb_gdp.

In [108]:
wb_gdp = pd.read_csv('../Datasets/Raw_datasets/GDP/world_bank_gdp_1960_to_2023.csv')

wb_gdp_2000_to_2015 = wb_gdp[['Country Name', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']]
wb_gdp_drop_na = wb_gdp_2000_to_2015.dropna().reset_index(drop=True)

Successfully loaded the world_bank_health_expend_percentage.csv as a Pandas dataframe in variable wb_health_exp_perc.

In [109]:
wb_health_exp_perc = pd.read_csv('../Datasets/Raw_datasets/Health_Expenditure/world_bank_health_expend_percentage.csv')
wb_exp_perc_2000_to_2015 = wb_health_exp_perc[['Country Name', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']]
wb_health_exp_perc_dropna = wb_exp_perc_2000_to_2015.dropna().reset_index(drop=True)

Successfully loaded the world_bank_population.csv as a Pandas dataframe in variable world_population.

In [110]:
world_life_expectancy= pd.read_csv('../Datasets/Raw_datasets/Life_expectancy/life_expectancy_from_world_bank.csv')


Successfully loaded the world_bank_population.csv as a Pandas dataframe in variable world_population.

In [111]:
education_expenditure= pd.read_csv('../Datasets/Raw_datasets/Education_expenditure/World_Bank_education_expenditure_gdp_percentage.csv')


Successfully loaded the mean_body_mass_index_bmi_in_adult_males.csv as a Pandas dataframe in variable world_in_data_bmi.

In [112]:
world_in_data_bmi= pd.read_csv('../Datasets/Raw_datasets/bmi/Male/mean_body_mass_index_bmi_in_adult_males.csv')
world_in_data_bmi_female = pd.read_csv('../Datasets/Raw_datasets/bmi/Female/mean_body_mass_index_bmi_in_adult_women.csv')

Successfully loaded the world_bank_unemployment_total.csv as a Pandas dataframe in variable unemployment_total.

In [113]:
unemployment_total = pd.read_csv('../Datasets/Raw_datasets/unemployment_total/world_bank_unemployment_total.csv')


Tidying life_expectancy:
Renaming all the columns to more usable variables. The columns are all either of type integer, float or object, making it suitbale for visualization

In [114]:
print(life_expectancy.columns)

life_expectancy.rename(columns= {
    'Country' : 'country',
    'Year' : 'year',
    'Status' : 'status',
    'Life expectancy ' : 'life_expectancy',
    'Adult Mortality' : 'adult_mortality',
    'infant deaths' : 'infant_deaths',
    'Alcohol' : 'alcohol',
    'percentage expenditure' : 'percentage_expenditure',
    'Hepatitis B' : 'hepatitis_b',
    'Measles ' : 'measles',
    ' BMI ' : 'bmi',
    'under-five deaths ' : 'under_five_deaths', 
    'Polio' : 'polio',
    'Total expenditure' : 'total_expenditure',
    'Diphtheria ' : 'diphtheria',
    ' HIV/AIDS' : 'hiv_aids',
    'GDP' : 'gdp',
    'Population' : 'population',
    ' thinness  1-19 years' : 'thinness_1_to_19_years',
    ' thinness 5-9 years' : 'thinness_5_to_9_years',
    'Income composition of resources' : 'income_composition_of_resources',
    'Schooling' : 'schooling'
}, inplace=True)

Index(['Country', 'Year', 'Status', 'Life expectancy ', 'Adult Mortality',
       'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       'Measles ', ' BMI ', 'under-five deaths ', 'Polio', 'Total expenditure',
       'Diphtheria ', ' HIV/AIDS', 'GDP', 'Population',
       ' thinness  1-19 years', ' thinness 5-9 years',
       'Income composition of resources', 'Schooling'],
      dtype='object')


Finding rows in life_expectancy dataframe of countries which only have one entry (particularly '2013') and deleting those rows.

In [115]:
# Create a dictionary of unique countries with their value counts
unique_countries = dict(life_expectancy.country.value_counts())

# Creat an array of all the countries which appears more than once
valid_countries = np.array([key for key, value in list(unique_countries.items()) if value > 1])

# Modifying countries in 'country' column so all countries which only appear once becomes a missing value.
# Dropping all rows where a nan value occurs in 'country' column
life_exp_valid_countries = life_expectancy.copy()
life_exp_valid_countries['country'] = life_exp_valid_countries['country'].apply(lambda x: x if x in valid_countries else np.nan)
life_exp_valid_countries = life_exp_valid_countries.dropna(subset=['country'])


Working with World Bank Population data and cleaning:
1.  Creating list of unique countries in both World Bank and Life Expectancy df
2.  Creating modification list of uniques life expectancy countries for removal if present in world bank gdp df. All world bank countries are named exactly the same.
3.  Using thefuzz.process lib matches can be found to be updated and stored to matches list for updating
4.  Some countries which were not correctly matched could be manually recorded and stored in new_matches_list
5.  All countries in world bank population are replaced with values matching that of life expectancy
6.  Checking countries not the same in both dfs for process assurance
7.  Three Countries appearing in life expectancy but not population df are dropped
8.  Format population df to only contain ddata for appropriate years


In [116]:
# Create lists of unique countries in both world bank population dataframe and Life Expectancy dataframe for comparison
country_list_wb = list(wb_gdp_2000_to_2015['Country Name'])
country_list_le = list(life_expectancy['country'][life_expectancy['year'] == 2000])

# Finding countries from Life Expectancy df which are not in same format as in world bank gdp dataframe
countries_to_modify = country_list_le.copy()
for country in country_list_le:
    if country in country_list_wb:
        countries_to_modify.remove(country)

# Finding matches in world bank population dataframe using process.extractOne funcion from thefuzz lib for renaming to country names in Life Expectancy Df 
# - First string is country for Life Exp df
# - second is world bank population df, 
# - third is match level percentage 
matches = []
for country in countries_to_modify:
    match = process.extractOne(country, country_list_wb, score_cutoff=80)
    if match:
        matches.append((country, match[0], match[1]))

# Retrieving all info from thefuzz output data and listing it to prepare for manual changes
new_matches_list = []
for tuple in matches:
    new_matches_list.append(list(tuple))

# Manually changing new matches list which was not matched correctly to correct country names
new_matches_list[4][1] = "Korea, Dem. People's Rep."
new_matches_list[5][1] = 'Congo, Rep.'
new_matches_list[8][1] = 'Iran, Islamic Rep.'
new_matches_list[11][1] = 'Korea, Rep.'
new_matches_list[13][1] = 'St. Lucia'
new_matches_list[15][1] = 'North Macedonia'
new_matches_list.append(['Turkey', 'Turkiye'])
new_matches_list.append(['Slovakia', 'Slovak Republic'])

# Iterating over matches and replacing old country names in world bank pop df with synchronized names occurring in life expectancy df
world_pop_cleaned = world_population.copy()
for world_pop_country in new_matches_list:
    world_pop_cleaned['Country Name'] = world_pop_cleaned['Country Name'].replace(world_pop_country[1], world_pop_country[0])


# Checking for countries occurring in life expectancy df and not in world bank pop df
country_list_wb_new = list(world_pop_cleaned['Country Name'])
country_list_le_1 = list(life_expectancy['country'][life_expectancy['year'] == 2000])


countries_cross_check = []
for country in country_list_le_1:
    if country not in country_list_wb_new:
        countries_cross_check.append(country)


# Dropping 'Dominican Republic', 'Kyrgyzstan' and 'Swaziland'
life_exp_clean = life_exp_valid_countries.drop(life_exp_valid_countries[(life_exp_valid_countries['country'] == countries_cross_check[0]) | (life_exp_valid_countries['country'] == countries_cross_check[1]) | (life_exp_valid_countries['country'] == countries_cross_check[2])].index).reset_index(drop=True)
life_exp_clean = life_exp_clean.sort_values(by=['country', 'year'], ascending=[True, False])
life_exp_clean_countries = life_exp_clean['country'].unique()

# Finding countries in life expectancy df which needs to be filled with population values from world bank population df
life_exp_pop_nan = life_exp_clean[life_exp_clean['gdp'].isna()]
countries_to_fill = life_exp_pop_nan['country'].unique()
#print(countries_to_fill)

# formatting world bank pop df to only include population values relavent to year 2000 to 2015 from life expectancy df 
world_pop_2000_to_2015 = world_pop_cleaned[['Country Name', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']]



Creating function for generating World Bank dataframes separately, by year and in the same format as life expectancy df, to be concattenated and then joined

In [117]:
# Finding all values for countries listed in life_expectancy_clean and formatting dataframe to be in same format for later concattenation
life_exp_clean_countries = life_exp_clean['country'].unique()
wb_pop_drop_na = world_pop_2000_to_2015.copy()
wb_pop_drop_na = wb_pop_drop_na.dropna().reset_index(drop=True)


# Checking that all country names in life_exp_clean is the same as in world_pop_cleaned. Only 'Dominican Republic' and 'Swaziland' are not in pop df so they will be removed
#   wb_crosscheck_countries = world_pop_cleaned['Country Name'].unique()
#   not_in_wb_pop_countries = []
#   for country in life_exp_clean_countries:
#       if country not in wb_crosscheck_countries:
#           not_in_wb_pop_countries.append(country)
#   print(not_in_wb_gdp_countries)


# Create a function to clean gdp df for each year to be concattenated and joined with life_expectancy df

# year_column = 'year' to filter
# target_column = World Bank target dataframe 'column' data targeted. In this case - 'population'
# columns_to_disply = list of columns to display ex ['country', 'year', 'population']

def modified_wb_pop_by_year(year_column, target_column, columns_to_display):
    wb_pop_year = wb_pop_drop_na.copy()
    wb_pop_year = wb_pop_year[['Country Name', year_column]]
    wb_pop_year_modified = wb_pop_year.copy()
    
    year_list = []
    for x in range(0, 265):
        year_list.append(int(year_column))
    wb_pop_year_modified['year'] = year_list
    
    wb_pop_year_modified.rename(columns={
        'Country Name' : 'country',
        year_column : target_column
    }, inplace=True)
    
    wb_pop_year_modified = wb_pop_year_modified[columns_to_display]

    # Create df to only display countries in wb_pop_year_modified which occur in life_exp_clean_countries
    wb_pop_year_modified['country'] = wb_pop_year_modified['country'].apply(lambda x: x if x in life_exp_clean_countries else pd.NA).reset_index(drop=True)
    wb_pop_year_modified = wb_pop_year_modified.dropna(subset=['country']).reset_index(drop=True)
    return wb_pop_year_modified



Concattenating all wb_gdp_year_modified to create and reorder dataframe to coincide with life expectancy df
1.  Using modified_wb_pop_by_year function dfs are generated by year
2.  All generated dfs are concattenated
3.  Concattenated df is correctly sorted
4.  Merging life_expectancy_clean and wb_pop_sorted so population in the former is overridden by population in the latter

In [118]:
# Running modified_wb_pop_by_year function for all years from 2000 to 2015 to create seperate dfs to modify life expectance df
wb_pop_2000_modified = modified_wb_pop_by_year('2000', 'population', ['country', 'year', 'population'])
wb_pop_2001_modified = modified_wb_pop_by_year('2001', 'population', ['country', 'year', 'population'])
wb_pop_2002_modified = modified_wb_pop_by_year('2002', 'population', ['country', 'year', 'population'])
wb_pop_2003_modified = modified_wb_pop_by_year('2003', 'population', ['country', 'year', 'population'])
wb_pop_2004_modified = modified_wb_pop_by_year('2004', 'population', ['country', 'year', 'population'])
wb_pop_2005_modified = modified_wb_pop_by_year('2005', 'population', ['country', 'year', 'population'])
wb_pop_2006_modified = modified_wb_pop_by_year('2006', 'population', ['country', 'year', 'population'])
wb_pop_2007_modified = modified_wb_pop_by_year('2007', 'population', ['country', 'year', 'population'])
wb_pop_2008_modified = modified_wb_pop_by_year('2008', 'population', ['country', 'year', 'population'])
wb_pop_2009_modified = modified_wb_pop_by_year('2009', 'population', ['country', 'year', 'population'])
wb_pop_2010_modified = modified_wb_pop_by_year('2010', 'population', ['country', 'year', 'population'])
wb_pop_2011_modified = modified_wb_pop_by_year('2011', 'population', ['country', 'year', 'population'])
wb_pop_2012_modified = modified_wb_pop_by_year('2012', 'population', ['country', 'year', 'population'])
wb_pop_2013_modified = modified_wb_pop_by_year('2013', 'population', ['country', 'year', 'population'])
wb_pop_2014_modified = modified_wb_pop_by_year('2014', 'population', ['country', 'year', 'population'])
wb_pop_2015_modified = modified_wb_pop_by_year('2015', 'population', ['country', 'year', 'population'])

# Concattenating population dfs for all years
wb_pop_2000_to_2015_modified = pd.concat([wb_pop_2000_modified, wb_pop_2001_modified, wb_pop_2002_modified, wb_pop_2003_modified, wb_pop_2004_modified, 
                                          wb_pop_2005_modified, wb_pop_2006_modified, wb_pop_2007_modified, wb_pop_2008_modified, wb_pop_2009_modified, 
                                          wb_pop_2010_modified, wb_pop_2011_modified, wb_pop_2012_modified, wb_pop_2013_modified, wb_pop_2014_modified, 
                                          wb_pop_2015_modified])

# Sorting new population dataframe in same format as Life Expectancy df
wb_pop_sorted = wb_pop_2000_to_2015_modified.sort_values(by=['country', 'year'], ascending=[True, False]).reset_index(drop=True)

wb_pop_array = wb_pop_sorted['population']
life_exp_clean['population'] = wb_pop_array

Modifying World Bank's gdp data using variables created whilst cleaning Population dataframe - Same procedure followed as modifying population to find yearly df (2000 - 2015)
1.  new_matches_list contains all the country names which needs to be replaced

In [119]:
# Iterating over matches and replacing old country names in world bank gdp df with synchronized names occurring in life expectancy df
world_gdp_cleaned = wb_gdp.copy()
for world_gdp_country in new_matches_list:
    world_gdp_cleaned['Country Name'] = world_gdp_cleaned['Country Name'].replace(world_gdp_country[1], world_gdp_country[0])


# formatting world bank gdp df to only include gdp values relavent to year 2000 to 2015 from life expectancy df 
world_gdp_2000_to_2015 = world_gdp_cleaned[['Country Name', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']]

# Finding all values for countries listed in life_expectancy_clean and formatting dataframe to be in same format for later concattenation
life_exp_clean_countries = life_exp_clean['country'].unique()
wb_gdp_drop_na = world_gdp_2000_to_2015.copy()
wb_gdp_drop_na = wb_gdp_drop_na.dropna().reset_index(drop=True)


# Create a function to clean gdp df for each year to be concattenated and joined with life_expectancy df

# year_column = 'year' to filter
# target_column = World Bank target dataframe 'column' data targeted
# columns_to_disply = list of columns to display ex ['country', 'year', 'gdp']

def modified_wb_gdp_by_year(year_column, target_column, columns_to_display):
    wb_gdp_year = wb_gdp_drop_na.copy()
    #print(wb_gdp_year)
    wb_gdp_year = wb_gdp_year[['Country Name', year_column]]
    wb_gdp_year_modified = wb_gdp_year.copy()
    
    year_list = []
    for x in range(0, 248):
        year_list.append(int(year_column))
    wb_gdp_year_modified['year'] = year_list
    
    wb_gdp_year_modified.rename(columns={
        'Country Name' : 'country',
        year_column : target_column
    }, inplace=True)
    
    wb_gdp_year_modified = wb_gdp_year_modified[columns_to_display]

    # Create df to only display countries in wb_gdp_year_modified which occur in life_exp_clean_countries
    wb_gdp_year_modified['country'] = wb_gdp_year_modified['country'].apply(lambda x: x if x in life_exp_clean_countries else pd.NA).reset_index(drop=True)
    wb_gdp_year_modified = wb_gdp_year_modified.dropna(subset=['country']).reset_index(drop=True)
    return wb_gdp_year_modified

# Running modified_wb_gdp_by_year function for all years from 2000 to 2015 to create seperate dfs to modify life expectance df
wb_gdp_2000_modified = modified_wb_gdp_by_year('2000', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2001_modified = modified_wb_gdp_by_year('2001', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2002_modified = modified_wb_gdp_by_year('2002', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2003_modified = modified_wb_gdp_by_year('2003', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2004_modified = modified_wb_gdp_by_year('2004', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2005_modified = modified_wb_gdp_by_year('2005', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2006_modified = modified_wb_gdp_by_year('2006', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2007_modified = modified_wb_gdp_by_year('2007', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2008_modified = modified_wb_gdp_by_year('2008', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2009_modified = modified_wb_gdp_by_year('2009', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2010_modified = modified_wb_gdp_by_year('2010', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2011_modified = modified_wb_gdp_by_year('2011', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2012_modified = modified_wb_gdp_by_year('2012', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2013_modified = modified_wb_gdp_by_year('2013', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2014_modified = modified_wb_gdp_by_year('2014', 'gdp', ['country', 'year', 'gdp'])
wb_gdp_2015_modified = modified_wb_gdp_by_year('2015', 'gdp', ['country', 'year', 'gdp'])


Concattenate dataframes created with modified_wb_gdp_by_year function. Finding unique countries in life_expectancy_clean df which do not occur in World Bank cleaned GDP df and removing them in order to eliminate missing values in the Life Expectancy 'gdp'column.

In [120]:
wb_gdp_2000_to_2015_modified = pd.concat([wb_gdp_2000_modified, wb_gdp_2001_modified, wb_gdp_2002_modified, wb_gdp_2003_modified, wb_gdp_2004_modified, 
                                          wb_gdp_2005_modified, wb_gdp_2006_modified, wb_gdp_2007_modified, wb_gdp_2008_modified, wb_gdp_2009_modified, 
                                          wb_gdp_2010_modified, wb_gdp_2011_modified, wb_gdp_2012_modified, wb_gdp_2013_modified, wb_gdp_2014_modified, 
                                          wb_gdp_2015_modified])
wb_gdp_sorted = wb_gdp_2000_to_2015_modified.sort_values(by=['country', 'year'], ascending=[True, False]).reset_index(drop=True)

# Countries in life expectancy df and not in gdp df are dropped
wb_gdp_unique_countries = wb_gdp_sorted['country'].unique()
list_life_exp_countries_to_remove = []
for country in life_exp_clean_countries:
    if country not in wb_gdp_unique_countries:
        list_life_exp_countries_to_remove.append(country)
life_exp_clean['country'] = life_exp_clean['country'].apply(lambda x: pd.NA if x in list_life_exp_countries_to_remove else x)
life_exp_clean = life_exp_clean.dropna(subset=['country']).reset_index(drop=True)

# life expectancy df gdp column replaced with cleaned gdp array from world bank gdp df
wb_gdp_array = wb_gdp_sorted['gdp']
life_exp_clean['gdp'] = wb_gdp_array

Cleaning World Bank's Health Expenditure Percentage data using variables created whilst cleaning Population dataframe 
1.  new_matches_list contains all the country names which needs to be replaced

In [121]:
life_exp_clean_countries = list(life_exp_clean['country'].unique())

# Iterating over matches and replacing old country names in world bank health_expenditure_percentage df with synchronized names occurring in life expectancy df
wb_health_exp_perc_clean = wb_health_exp_perc.copy()
for country in new_matches_list:
    wb_health_exp_perc_clean['Country Name'] = wb_health_exp_perc_clean['Country Name'].replace(country[1], country[0])
wb_health_exp_perc_clean = wb_health_exp_perc_clean.sort_values(by=['Country Name'], ascending=[True]).reset_index(drop=True)

# formatting world bank health_expenditure_percentage df to only include health_expenditure_percentage values relavent to year 2000 to 2015 from life expectancy df 
world_h_e_p_2000_to_2015_raw = wb_health_exp_perc_clean[['Country Name', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']]

# Finding all values for countries listed in life_expectancy_clean and formatting dataframe to be in same format for later concattenation
health_exp_perc_clean_countries = list(world_h_e_p_2000_to_2015_raw['Country Name'].unique())

world_h_e_p_2000_to_2015 = world_h_e_p_2000_to_2015_raw.copy()
world_h_e_p_2000_to_2015['Country Name'] = world_h_e_p_2000_to_2015['Country Name'].apply(lambda x: x if x in life_exp_clean_countries else pd.NA )
wb_health_exp_perc_clean = world_h_e_p_2000_to_2015.dropna(subset=['Country Name']).reset_index(drop=True)


# Create a function to clean health_expenditure_percentage df for each year to be concattenated and joined with life_expectancy df

# year_column = 'year' to filter
# target_column = World Bank target dataframe 'column' data targeted
# columns_to_disply = list of columns to display ex ['country', 'year', 'health_expenditure_percentage']

def modified_wb_hep_by_year(year_column, target_column, columns_to_display):
    wb_health_expenditure_percentage_year = wb_health_exp_perc_clean.copy()
    #print(wb_health_expenditure_percentage_year)
    wb_health_expenditure_percentage_year = wb_health_expenditure_percentage_year[['Country Name', year_column]]
    wb_health_expenditure_percentage_year_mod = wb_health_expenditure_percentage_year.copy()
    
    year_list = []
    for x in range(0, 176):
        year_list.append(int(year_column))
    wb_health_expenditure_percentage_year_mod['year'] = year_list
    
    wb_health_expenditure_percentage_year_mod.rename(columns={
        'Country Name' : 'country',
        year_column : target_column
    }, inplace=True)
    
    wb_health_expenditure_percentage_year_mod = wb_health_expenditure_percentage_year_mod[columns_to_display]

    # Create df to only display countries in wb_health_expenditure_percentage_year_mod which occur in life_exp_clean_countries
    wb_health_expenditure_percentage_year_mod['country'] = wb_health_expenditure_percentage_year_mod['country'].apply(lambda x: x if x in health_exp_perc_clean_countries else pd.NA).reset_index(drop=True)
    wb_health_expenditure_percentage_year_mod = wb_health_expenditure_percentage_year_mod.dropna(subset=['country']).reset_index(drop=True)
    return wb_health_expenditure_percentage_year_mod

# Running modified_wb_hep_by_year function for all years from 2000 to 2015 to create seperate dfs to modify life expectance df
wb_health_expenditure_percentage_2000_modified = modified_wb_hep_by_year('2000', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2001_modified = modified_wb_hep_by_year('2001', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2002_modified = modified_wb_hep_by_year('2002', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2003_modified = modified_wb_hep_by_year('2003', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2004_modified = modified_wb_hep_by_year('2004', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2005_modified = modified_wb_hep_by_year('2005', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2006_modified = modified_wb_hep_by_year('2006', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2007_modified = modified_wb_hep_by_year('2007', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2008_modified = modified_wb_hep_by_year('2008', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2009_modified = modified_wb_hep_by_year('2009', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2010_modified = modified_wb_hep_by_year('2010', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2011_modified = modified_wb_hep_by_year('2011', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2012_modified = modified_wb_hep_by_year('2012', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2013_modified = modified_wb_hep_by_year('2013', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2014_modified = modified_wb_hep_by_year('2014', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])
wb_health_expenditure_percentage_2015_modified = modified_wb_hep_by_year('2015', 'health_expenditure_percentage', ['country', 'year', 'health_expenditure_percentage'])

# Concattenated all world bank health expenditure (by year) dfs and sorting to life exp clean df format:
wb_hep_2000_to_2015_modified = pd.concat([wb_health_expenditure_percentage_2000_modified, wb_health_expenditure_percentage_2001_modified, wb_health_expenditure_percentage_2002_modified, wb_health_expenditure_percentage_2003_modified, wb_health_expenditure_percentage_2004_modified, wb_health_expenditure_percentage_2005_modified, 
                                          wb_health_expenditure_percentage_2006_modified, wb_health_expenditure_percentage_2007_modified, wb_health_expenditure_percentage_2008_modified, wb_health_expenditure_percentage_2009_modified, wb_health_expenditure_percentage_2010_modified, 
                                          wb_health_expenditure_percentage_2011_modified, wb_health_expenditure_percentage_2012_modified, wb_health_expenditure_percentage_2013_modified, wb_health_expenditure_percentage_2014_modified, wb_health_expenditure_percentage_2015_modified, 
                                          ])
wb_hep_sorted = wb_hep_2000_to_2015_modified.sort_values(by=['country', 'year'], ascending=[True, False]).reset_index(drop=True)

# adding 'health_expenditure_percentage' column to life exp clean df
wb_hep_array = wb_hep_sorted['health_expenditure_percentage']
life_exp_clean['health_expenditure_percentage'] = wb_hep_array

# drop 'percentage_expenditure' column countaining incorrect values
# life_exp_clean = life_exp_clean.drop(columns=['percentage_expenditure'])


Cleaning World Bank's Health Expectany data using variables created whilst cleaning Population dataframe 
1.  new_matches_list contains all the country names which needs to be replaced

In [122]:
life_exp_clean_countries = list(life_exp_clean['country'].unique())

# Iterating over matches and replacing old country names in world bank life_expectancy df with synchronized names occurring in life expectancy df
wb_health_expectancy_clean = world_life_expectancy.copy()
for country in new_matches_list:
    wb_health_expectancy_clean['Country Name'] = wb_health_expectancy_clean['Country Name'].replace(country[1], country[0])
wb_health_expectancy_clean = wb_health_expectancy_clean.sort_values(by=['Country Name'], ascending=[True]).reset_index(drop=True)

# formatting world bank life_expectancy df to only include life_expectancy values relavent to year 2000 to 2015 from life expectancy df 
world_health_exp_2000_to_2015_raw = wb_health_expectancy_clean[['Country Name', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']]

# Finding all values for countries listed in life_expectancy_clean and formatting dataframe to be in same format for later concattenation
    # life_exp_clean_countries = list(world_health_exp_2000_to_2015_raw['Country Name'].unique())

world_health_exp_2000_to_2015 = world_health_exp_2000_to_2015_raw.copy()
world_health_exp_2000_to_2015['Country Name'] = world_health_exp_2000_to_2015['Country Name'].apply(lambda x: x if x in life_exp_clean_countries else pd.NA )
wb_health_expectancy_clean = world_health_exp_2000_to_2015.dropna(subset=['Country Name']).reset_index(drop=True)


# Create a function to clean life_expectancy df for each year to be concattenated and joined with life_expectancy df

# year_column = 'year' to filter
# target_column = World Bank target dataframe 'column' data targeted
# columns_to_disply = list of columns to display ex ['country', 'year', 'life_expectancy']

def modified_wb_health_exp_by_year(year_column, target_column, columns_to_display):
    wb_health_exp_year = wb_health_expectancy_clean.copy()
    #print(wb_health_exp_year)
    wb_health_exp_year = wb_health_exp_year[['Country Name', year_column]]
    wb_health_exp_year_mod = wb_health_exp_year.copy()
    
    year_list = []
    for x in range(0, 176):
        year_list.append(int(year_column))
    wb_health_exp_year_mod['year'] = year_list
    
    wb_health_exp_year_mod.rename(columns={
        'Country Name' : 'country',
        year_column : target_column
    }, inplace=True)
    
    wb_health_exp_year_mod = wb_health_exp_year_mod[columns_to_display]

    # Create df to only display countries in wb_health_exp_year_mod which occur in life_exp_clean_countries
    wb_health_exp_year_mod['country'] = wb_health_exp_year_mod['country'].apply(lambda x: x if x in life_exp_clean_countries else pd.NA).reset_index(drop=True)
    wb_health_exp_year_mod = wb_health_exp_year_mod.dropna(subset=['country']).reset_index(drop=True)
    return wb_health_exp_year_mod

# Running modified_wb_health_exp_by_year function for all years from 2000 to 2015 to create seperate dfs to modify life expectance df
wb_life_expectancy_2000_modified = modified_wb_health_exp_by_year('2000', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2001_modified = modified_wb_health_exp_by_year('2001', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2002_modified = modified_wb_health_exp_by_year('2002', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2003_modified = modified_wb_health_exp_by_year('2003', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2004_modified = modified_wb_health_exp_by_year('2004', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2005_modified = modified_wb_health_exp_by_year('2005', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2006_modified = modified_wb_health_exp_by_year('2006', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2007_modified = modified_wb_health_exp_by_year('2007', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2008_modified = modified_wb_health_exp_by_year('2008', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2009_modified = modified_wb_health_exp_by_year('2009', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2010_modified = modified_wb_health_exp_by_year('2010', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2011_modified = modified_wb_health_exp_by_year('2011', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2012_modified = modified_wb_health_exp_by_year('2012', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2013_modified = modified_wb_health_exp_by_year('2013', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2014_modified = modified_wb_health_exp_by_year('2014', 'life_expectancy', ['country', 'year', 'life_expectancy'])
wb_life_expectancy_2015_modified = modified_wb_health_exp_by_year('2015', 'life_expectancy', ['country', 'year', 'life_expectancy'])

# Concattenated all world bank health expenditure (by year) dfs and sorting to life exp clean df format:
wb_le_2000_to_2015_modified = pd.concat([wb_life_expectancy_2000_modified, wb_life_expectancy_2001_modified, wb_life_expectancy_2002_modified, wb_life_expectancy_2003_modified, wb_life_expectancy_2004_modified, wb_life_expectancy_2005_modified, 
                                          wb_life_expectancy_2006_modified, wb_life_expectancy_2007_modified, wb_life_expectancy_2008_modified, wb_life_expectancy_2009_modified, wb_life_expectancy_2010_modified, 
                                          wb_life_expectancy_2011_modified, wb_life_expectancy_2012_modified, wb_life_expectancy_2013_modified, wb_life_expectancy_2014_modified, wb_life_expectancy_2015_modified, 
                                          ])
wb_le_sorted = wb_le_2000_to_2015_modified.sort_values(by=['country', 'year'], ascending=[True, False]).reset_index(drop=True)

# adding 'life_expectancy' column to life exp clean df
wb_le_array = wb_le_sorted['life_expectancy']
life_exp_clean['life_expectancy'] = wb_le_array



Cleaning World Bank's Unemployment, total (% of total labor force) (modeled ILO estimate) data using variables created whilst cleaning Population dataframe 
1.  new_matches_list contains all the country names which needs to be replaced

In [123]:
life_exp_clean_countries = list(life_exp_clean['country'].unique())

# Iterating over matches and replacing old country names in world bank unemployment_total df with synchronized names occurring in life expectancy df
unemployed_total = unemployment_total.copy()
for country in new_matches_list:
    unemployed_total['Country Name'] = unemployed_total['Country Name'].replace(country[1], country[0])
unemployed_total = unemployed_total.sort_values(by=['Country Name'], ascending=[True]).reset_index(drop=True)

# formatting world bank unemployment_total df to only include unemployment_total values relavent to year 2000 to 2015 from world bank unemployment total df 
world_unemployment_2000_to_2015_raw = unemployed_total[['Country Name', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']]

# Finding all values for countries listed in unemployment_total_clean and formatting dataframe to be in same format for later concattenation
    # life_exp_clean_countries = list(world_unemployment_2000_to_2015_raw['Country Name'].unique())

world_unemployed_2000_to_2015 = world_unemployment_2000_to_2015_raw.copy()
world_unemployed_2000_to_2015['Country Name'] = world_unemployed_2000_to_2015['Country Name'].apply(lambda x: x if x in life_exp_clean_countries else pd.NA )
unemployed_total = world_unemployed_2000_to_2015.dropna(subset=['Country Name']).reset_index(drop=True)


# Create a function to clean unemployment_total df for each year to be concattenated and joined with unemployment_total df

# year_column = 'year' to filter
# target_column = World Bank target dataframe 'column' data targeted
# columns_to_disply = list of columns to display ex ['country', 'year', 'unemployment_total']

def modified_wb_unemployment_total_by_year(year_column, target_column, columns_to_display):
    wb_unemployed_year = unemployed_total.copy()
    #print(wb_unemployed_year)
    wb_unemployed_year = wb_unemployed_year[['Country Name', year_column]]
    wb_unemployed_year_mod = wb_unemployed_year.copy()
    
    year_list = []
    for x in range(0, 176):
        year_list.append(int(year_column))
    wb_unemployed_year_mod['year'] = year_list
    
    wb_unemployed_year_mod.rename(columns={
        'Country Name' : 'country',
        year_column : target_column
    }, inplace=True)
    
    wb_unemployed_year_mod = wb_unemployed_year_mod[columns_to_display]

    # Create df to only display countries in wb_unemployed_year_mod which occur in life_exp_clean_countries
    wb_unemployed_year_mod['country'] = wb_unemployed_year_mod['country'].apply(lambda x: x if x in life_exp_clean_countries else pd.NA).reset_index(drop=True)
    wb_unemployed_year_mod = wb_unemployed_year_mod.dropna(subset=['country']).reset_index(drop=True)
    return wb_unemployed_year_mod

# Running modified_wb_unemployment_total_by_year function for all years from 2000 to 2015 to create seperate dfs to modify life expectance df
wb_unemployment_total_2000_modified = modified_wb_unemployment_total_by_year('2000', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2001_modified = modified_wb_unemployment_total_by_year('2001', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2002_modified = modified_wb_unemployment_total_by_year('2002', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2003_modified = modified_wb_unemployment_total_by_year('2003', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2004_modified = modified_wb_unemployment_total_by_year('2004', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2005_modified = modified_wb_unemployment_total_by_year('2005', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2006_modified = modified_wb_unemployment_total_by_year('2006', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2007_modified = modified_wb_unemployment_total_by_year('2007', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2008_modified = modified_wb_unemployment_total_by_year('2008', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2009_modified = modified_wb_unemployment_total_by_year('2009', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2010_modified = modified_wb_unemployment_total_by_year('2010', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2011_modified = modified_wb_unemployment_total_by_year('2011', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2012_modified = modified_wb_unemployment_total_by_year('2012', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2013_modified = modified_wb_unemployment_total_by_year('2013', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2014_modified = modified_wb_unemployment_total_by_year('2014', 'unemployment_total', ['country', 'year', 'unemployment_total'])
wb_unemployment_total_2015_modified = modified_wb_unemployment_total_by_year('2015', 'unemployment_total', ['country', 'year', 'unemployment_total'])

# Concattenated all world bank health expenditure (by year) dfs and sorting to life exp clean df format:
wb_unemployed_2000_to_2015_modified = pd.concat([wb_unemployment_total_2000_modified, wb_unemployment_total_2001_modified, wb_unemployment_total_2002_modified, wb_unemployment_total_2003_modified, wb_unemployment_total_2004_modified, wb_unemployment_total_2005_modified, 
                                          wb_unemployment_total_2006_modified, wb_unemployment_total_2007_modified, wb_unemployment_total_2008_modified, wb_unemployment_total_2009_modified, wb_unemployment_total_2010_modified, 
                                          wb_unemployment_total_2011_modified, wb_unemployment_total_2012_modified, wb_unemployment_total_2013_modified, wb_unemployment_total_2014_modified, wb_unemployment_total_2015_modified, 
                                          ])
wb_unemployment_sorted = wb_unemployed_2000_to_2015_modified.sort_values(by=['country', 'year'], ascending=[True, False]).reset_index(drop=True)

# adding 'unemployment_total' column to life exp clean df
wb_unemployed_array = wb_unemployment_sorted['unemployment_total']
life_exp_clean['unemployment_total'] = wb_unemployed_array



Cleaning World Bank's Education Expenditure as per GDP per capita % data using variables created whilst cleaning Population dataframe 
1.  new_matches_list contains all the country names which needs to be replaced

In [124]:
life_exp_clean_countries = list(life_exp_clean['country'].unique())

# Iterating over matches and replacing old country names in world bank education_per_gdp df with synchronized names occurring in life expectancy df
education_per_gdp = education_expenditure.copy()
for country in new_matches_list:
    education_per_gdp['Country Name'] = education_per_gdp['Country Name'].replace(country[1], country[0])
education_per_gdp = education_per_gdp.sort_values(by=['Country Name'], ascending=[True]).reset_index(drop=True)

# formatting world bank education_per_gdp df to only include education_per_gdp values relavent to year 2000 to 2015 from world bank unemployment total df 
world_education_2000_to_2015_raw = education_per_gdp[['Country Name', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']]

# Finding all values for countries listed in education_per_gdp_clean and formatting dataframe to be in same format for later concattenation
    # life_exp_clean_countries = list(world_education_2000_to_2015_raw['Country Name'].unique())

world_unemployed_2000_to_2015 = world_education_2000_to_2015_raw.copy()
world_unemployed_2000_to_2015['Country Name'] = world_unemployed_2000_to_2015['Country Name'].apply(lambda x: x if x in life_exp_clean_countries else pd.NA )
education_per_gdp = world_unemployed_2000_to_2015.dropna(subset=['Country Name']).reset_index(drop=True)


# Create a function to clean education_per_gdp df for each year to be concattenated and joined with education_per_gdp df

# year_column = 'year' to filter
# target_column = World Bank target dataframe 'column' data targeted
# columns_to_disply = list of columns to display ex ['country', 'year', 'education_per_gdp']

def modified_wb_education_per_gdp_by_year(year_column, target_column, columns_to_display):
    wb_education_year = education_per_gdp.copy()
    #print(wb_education_year)
    wb_education_year = wb_education_year[['Country Name', year_column]]
    wb_education_year_mod = wb_education_year.copy()
    
    year_list = []
    for x in range(0, 176):
        year_list.append(int(year_column))
    wb_education_year_mod['year'] = year_list
    
    wb_education_year_mod.rename(columns={
        'Country Name' : 'country',
        year_column : target_column
    }, inplace=True)
    
    wb_education_year_mod = wb_education_year_mod[columns_to_display]

    # Create df to only display countries in wb_education_year_mod which occur in life_exp_clean_countries
    wb_education_year_mod['country'] = wb_education_year_mod['country'].apply(lambda x: x if x in life_exp_clean_countries else pd.NA).reset_index(drop=True)
    wb_education_year_mod = wb_education_year_mod.dropna(subset=['country']).reset_index(drop=True)
    return wb_education_year_mod

# Running modified_wb_education_per_gdp_by_year function for all years from 2000 to 2015 to create seperate dfs to modify life expectance df
wb_education_per_gdp_2000_modified = modified_wb_education_per_gdp_by_year('2000', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2001_modified = modified_wb_education_per_gdp_by_year('2001', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2002_modified = modified_wb_education_per_gdp_by_year('2002', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2003_modified = modified_wb_education_per_gdp_by_year('2003', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2004_modified = modified_wb_education_per_gdp_by_year('2004', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2005_modified = modified_wb_education_per_gdp_by_year('2005', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2006_modified = modified_wb_education_per_gdp_by_year('2006', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2007_modified = modified_wb_education_per_gdp_by_year('2007', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2008_modified = modified_wb_education_per_gdp_by_year('2008', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2009_modified = modified_wb_education_per_gdp_by_year('2009', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2010_modified = modified_wb_education_per_gdp_by_year('2010', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2011_modified = modified_wb_education_per_gdp_by_year('2011', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2012_modified = modified_wb_education_per_gdp_by_year('2012', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2013_modified = modified_wb_education_per_gdp_by_year('2013', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2014_modified = modified_wb_education_per_gdp_by_year('2014', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])
wb_education_per_gdp_2015_modified = modified_wb_education_per_gdp_by_year('2015', 'education_per_gdp', ['country', 'year', 'education_per_gdp'])

# Concattenated all world bank health expenditure (by year) dfs and sorting to life exp clean df format:
wb_education_per_gdp_2000_to_2015_modified = pd.concat([wb_education_per_gdp_2000_modified, wb_education_per_gdp_2001_modified, wb_education_per_gdp_2002_modified, wb_education_per_gdp_2003_modified, wb_education_per_gdp_2004_modified, wb_education_per_gdp_2005_modified, 
                                          wb_education_per_gdp_2006_modified, wb_education_per_gdp_2007_modified, wb_education_per_gdp_2008_modified, wb_education_per_gdp_2009_modified, wb_education_per_gdp_2010_modified, 
                                          wb_education_per_gdp_2011_modified, wb_education_per_gdp_2012_modified, wb_education_per_gdp_2013_modified, wb_education_per_gdp_2014_modified, wb_education_per_gdp_2015_modified, 
                                          ])
wb_education_per_gdp_sorted = wb_education_per_gdp_2000_to_2015_modified.sort_values(by=['country', 'year'], ascending=[True, False]).reset_index(drop=True)

# adding 'education_per_gdp' column to life exp clean df
wb_education_per_gdp_array = wb_education_per_gdp_sorted['education_per_gdp']
life_exp_clean['education_per_gdp'] = wb_education_per_gdp_array



Working with Our World in Data BMI data and cleaning:
1.  Creating list of unique countries in both Our World in Data and Life Expectancy df
2.  Creating modification list of uniques life expectancy countries for removal if present in Our World in Data male bmi df. 
3.  Using thefuzz.process lib matches can be found to be updated and stored to matches list for updating
4.  Some countries which were not correctly matched could be manually recorded and stored in new_matches_list
5.  All countries in Our World in Data are replaced with values matching that of life expectancy
6.  Checking countries not the same in both dfs for process assurance
7.  Changing life expectancy bmi column to bmi_male
8. Adding correct values to life expectancy bmi_male column


In [125]:
# Create lists of unique countries in both our world in data bmi dataframe and Life Expectancy dataframe for comparison
country_list_owid = list(world_in_data_bmi['Entity'])
country_list_le = list(life_exp_clean['country'].unique())

# Finding countries from Life Expectancy df which are not in same format as in our world in data bmi dataframe
countries_to_modify = country_list_le.copy()
for country in country_list_le:
    if country in country_list_owid:
        countries_to_modify.remove(country)

# Finding matches in our world in data bmi dataframe using process.extractOne funcion from thefuzz lib for renaming to country names in Life Expectancy Df 
# - First string is country for Life Exp df
# - second is our world in data bmi df, 
# - third is match level percentage 
matches = []
for country in countries_to_modify:
    match = process.extractOne(country, country_list_owid, score_cutoff=80)
    if match:
        matches.append((country, match[0], match[1]))

# Retrieving all info from thefuzz output data and listing it to prepare for manual changes
new_matches_list = []
for tuple in matches:
    new_matches_list.append(list(tuple))

# Manually changing new matches list which was not matched correctly to correct country names
new_matches_list[6][1] =  'Laos'
new_matches_list[8][1] =  'South Korea'
new_matches_list[12][1] = 'North Macedonia'
new_matches_list[14][1] = 'United Kingdom'



# Iterating over matches and replacing old country names in our world in data bmi df with synchronized names occurring in life expectancy df
world_id_bmi_cleaned = world_in_data_bmi.copy()
for country in new_matches_list:
    world_id_bmi_cleaned['Entity'] = world_id_bmi_cleaned['Entity'].replace(country[1], country[0])


# Checking for countries occurring in our world in data bmi df and not in life expectancy df
country_list_owid_new = list(world_id_bmi_cleaned['Entity'].unique())
country_list_le_1 = list(life_exp_clean['country'].unique())


countries_cross_check = []
for country in country_list_owid_new:
    if country not in country_list_le_1:
        countries_cross_check.append(country)

# Drop countries ('Entity') in our world in data df which do not occur in life expectancy df so both are in sync
world_id_bmi_cleaned['Entity'] = world_id_bmi_cleaned['Entity'].apply(lambda x: pd.NA if x in countries_cross_check else x)
world_id_bmi_cleaned = world_id_bmi_cleaned.dropna(subset=['Entity']).reset_index(drop=True)

# Drop 'Code' column in our world in data df
world_id_bmi_cleaned = world_id_bmi_cleaned.drop(columns=['Code']).sort_values(by=['Entity', 'Year'], ascending=[True, False]).reset_index(drop=True)

# Change 'bmi' column to 'bmi_male' in life expectancy df
life_exp_clean = life_exp_clean.rename(columns={
    'bmi' : 'bmi_male'
})

# Replace incorrect bmi values in life expectancy df with correct values from our world in data
life_exp_clean['bmi_male'] = world_id_bmi_cleaned['Mean BMI (male)']


Using exact same format as converting bmi data to insert bmi_female values from our world in data df to life expectancy bmi_female column

In [126]:
# Create lists of unique countries in both our world in data bmi dataframe and Life Expectancy dataframe for comparison
country_list_owid = list(world_in_data_bmi_female['Entity'])
country_list_le = list(life_exp_clean['country'].unique())

# Finding countries from Life Expectancy df which are not in same format as in our world in data bmi dataframe
countries_to_modify = country_list_le.copy()
for country in country_list_le:
    if country in country_list_owid:
        countries_to_modify.remove(country)

# Finding matches in our world in data bmi dataframe using process.extractOne funcion from thefuzz lib for renaming to country names in Life Expectancy Df 
# - First string is country for Life Exp df
# - second is our world in data bmi df, 
# - third is match level percentage 
matches = []
for country in countries_to_modify:
    match = process.extractOne(country, country_list_owid, score_cutoff=80)
    if match:
        matches.append((country, match[0], match[1]))

# Retrieving all info from thefuzz output data and listing it to prepare for manual changes
new_matches_list = []
for tuple in matches:
    new_matches_list.append(list(tuple))

# Manually changing new matches list which was not matched correctly to correct country names
new_matches_list[6][1] =  'Laos'
new_matches_list[8][1] =  'South Korea'
new_matches_list[12][1] = 'North Macedonia'
new_matches_list[14][1] = 'United Kingdom'



# Iterating over matches and replacing old country names in our world in data bmi df with synchronized names occurring in life expectancy df
world_id_bmi_female_cleaned = world_in_data_bmi_female.copy()
for country in new_matches_list:
    world_id_bmi_female_cleaned['Entity'] = world_id_bmi_female_cleaned['Entity'].replace(country[1], country[0])


# Checking for countries occurring in our world in data bmi df and not in life expectancy df
country_list_owid_new = list(world_id_bmi_female_cleaned['Entity'].unique())
country_list_le_1 = list(life_exp_clean['country'].unique())


countries_cross_check = []
for country in country_list_owid_new:
    if country not in country_list_le_1:
        countries_cross_check.append(country)

# Drop countries ('Entity') in our world in data df which do not occur in life expectancy df so both are in sync
world_id_bmi_female_cleaned['Entity'] = world_id_bmi_female_cleaned['Entity'].apply(lambda x: pd.NA if x in countries_cross_check else x)
world_id_bmi_female_cleaned = world_id_bmi_female_cleaned.dropna(subset=['Entity']).reset_index(drop=True)

# Drop 'Code' column in our world in data df
world_id_bmi_female_cleaned = world_id_bmi_female_cleaned.drop(columns=['Code']).sort_values(by=['Entity', 'Year'], ascending=[True, False]).reset_index(drop=True)

# Create new column for female bmi values
life_exp_clean['bmi_female'] = world_id_bmi_female_cleaned['Mean BMI (female)']


Create first cleaned dataset for initial descriptive, exploritory and inferential data analysis; then performing data visualization.
Following variables included:
1.  country - 176 Countries
2.  year - 2000 to 2015
3.  status - Developing/Developed
4.  life_expectancy - Probability of life expectancy at birth
5.  bmi - Average BMI of entire population
6.  gdp - Gross Demestic Product per Capita
7.  population - Population of country


In [127]:
life_exp_clean_2 = life_exp_clean.copy()
life_exp_clean_2 = life_exp_clean_2[['country', 'year', 'status', 'life_expectancy', 'bmi_male', 'bmi_female', 'gdp', 'population']]


Creating csv file in cleaned_datasets folder of first cleaned dataset from life_exp_clean_2 with no missing values. Validity of data in columns - bmi, adult_mortality, under_five_deaths, mealses and hiv_aids still need to be checked in WHO and World Bank datasets and modified if values are incorrect. For now I will use this data for analysis and visualization.

In [128]:
life_exp_clean_2.to_csv("../Datasets/Cleaned_datasets/Life_expectancy_cleaned_1.csv")